# 02 — Export and validate block 0
Use the original multi-prompt static PT2E/LiteRT conversion path. Each fresh execution writes
a new model under `artifacts/models/reexports/<run-id>/`, validation inputs under
`artifacts/exports/reexports/<run-id>/`, and metrics/manifest under `results/notebook_runs/<run-id>/`.
The frozen baseline model is preserved. The export contains INT8 matrix operators and remaining
floating-point operators; it is a host-side reference, not a complete integer block.
The wrapper has no KV-cache input and no external padding mask. Padded tests report both
all-position and real-token errors. Original calibration prompt strings are retained.

Start from a fresh kernel and run top to bottom. The final cell records output hashes; incomplete runs remain `started`.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the repository root or notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/python/workflow_config.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start Jupyter in the extracted tinystories-rv64imc folder.')
sys.path.insert(0, str(ROOT / 'src/python'))
from workflow_config import (MODEL_ID, MODEL_REVISION, SEQUENCE_LENGTH,
                             MODEL_DIR, EXPORT_DIR, RESULT_DIR, C_DIR, MODEL_PATH,
                             PROMPT, GENERATED_LENGTH)
from notebook_runs import start_run, finish_run
RUN_DIR = start_run(ROOT, 'notebooks/02_export_block0.ipynb', {'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'sequence_length': SEQUENCE_LENGTH})
RESULT_DIR = RUN_DIR
MODEL_DIR = ROOT / 'artifacts/models/reexports' / RUN_DIR.name
MODEL_PATH = MODEL_DIR / 'block0_w8a8.tflite'
EXPORT_DIR = ROOT / 'artifacts/exports/reexports' / RUN_DIR.name


In [ ]:
import json
import hashlib
import importlib.metadata as metadata
import platform
import numpy as np
import torch
from torch import nn
from transformers import AutoModelForCausalLM, AutoTokenizer
import litert_torch
from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e
from litert_torch.quantize.pt2e_quantizer import PT2EQuantizer, get_symmetric_quantization_config
from litert_torch.quantize.quant_config import QuantConfig

torch.manual_seed(42)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, attn_implementation='eager').cpu().eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
for path in [MODEL_DIR, EXPORT_DIR, RESULT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## Create the block input from token and position embeddings

In [ ]:
def create_block0_input(model, input_ids):
    batch_size, sequence_length = input_ids.shape

    position_ids = torch.arange(
        sequence_length,
        dtype=torch.long,
        device=input_ids.device,
    ).unsqueeze(0)

    with torch.no_grad():
        token_embeddings = model.transformer.wte(
            input_ids
        )

        position_embeddings = model.transformer.wpe(
            position_ids
        )

        block_input = model.transformer.drop(
            token_embeddings + position_embeddings
        )

    return block_input

In [ ]:
encoded = tokenizer(PROMPT, add_special_tokens=False, return_tensors='pt')
if encoded['input_ids'].shape[1] < SEQUENCE_LENGTH:
    raise ValueError('The benchmark prompt must contain at least 32 tokens.')
input_ids = encoded['input_ids'][:, :SEQUENCE_LENGTH]
block_input = create_block0_input(model, input_ids)
assert tuple(block_input.shape) == (1, SEQUENCE_LENGTH, 128)
print('Block input:', block_input.shape, block_input.dtype)

In [ ]:
class BlockWrapper(nn.Module):
    def __init__(self, transformer_block):
        super().__init__()

        self.transformer_block = transformer_block

    def forward(self, hidden_states):
        block_outputs = self.transformer_block(
            hidden_states,
            layer_past=None,
            attention_mask=None,
            use_cache=False,
            output_attentions=False,
        )

        output_hidden_states = block_outputs[0]

        return output_hidden_states

In [ ]:
block0 = BlockWrapper(
    model.transformer.h[0]
)

block0.eval()

In [ ]:
def export_block(
    block,
    example_input,
):
    block.eval()

    example_arguments = (
        example_input.detach(),
    )
    print(type(example_input))
    print(type(example_arguments))

    exported_program = torch.export.export(
        block,
        example_arguments,
    )

    return exported_program

## Check the FP32 exported graph

In [ ]:
exported = export_block(block0, block_input)
with torch.no_grad():
    fp32_example = block0(block_input)
    exported_example = exported.module()(block_input)
torch.testing.assert_close(exported_example, fp32_example)
print('FP32 torch.export check passed')

# Optional preliminary conversion from testoneTransformer.ipynb.
EXPORT_FP32_TFLITE = False
if EXPORT_FP32_TFLITE:
    litert_torch.convert(block0, (block_input,)).export(str(MODEL_DIR / 'block0_fp32.tflite'))

## Calibrate once using the original eight prompts
Keep these exact strings, including whitespace, when comparing with the old export.
The original quantizer requested per-tensor static quantization. The converted graph
may store repeated scales for each output channel; inspect its actual metadata in notebook 03.

In [ ]:
from litert_torch.quantize.pt2e_quantizer import (
    PT2EQuantizer,
    get_symmetric_quantization_config,
)

def create_w8a8_quantizer():
    quantization_config = get_symmetric_quantization_config(
        is_per_channel=False,
        is_dynamic=False,
        is_qat=False,
    )

    quantizer = PT2EQuantizer().set_global(
        quantization_config
    )

    return quantizer


quantizer = create_w8a8_quantizer()
print(quantizer)

In [ ]:
calibration_prompts = [
    """
    Once upon a time, a little girl named Lily lived beside a quiet
    forest. Every morning she carried bread to her grandmother and
    greeted all the animals along the path.
    """,

    """
    Tom found a small red ball underneath his bed. He took it outside
    and played with his dog until the sun disappeared behind the hills.
    """,

    """
    A young rabbit wanted to cross the river, but the water was too
    deep. His friends gathered branches and helped him build a bridge.
    """,

    """
    Mia planted a tiny seed in the garden. She watered it every day,
    and soon a tall sunflower grew beside her bedroom window.
    """,

    """
    The little bird could not fly as high as the others. She practiced
    every afternoon until she finally reached the top of the old tree.
    """,

    """
    Ben heard a strange sound inside the kitchen. He opened the door
    slowly and discovered a hungry kitten looking for some food.
    """,

    """
    A kind farmer shared his vegetables with the people in his village.
    When winter arrived, everyone helped him repair his wooden barn.
    """,

    """
    Sara lost her favorite blue hat at the park. She searched near the
    swings and found it beside a sleeping brown puppy.
    """,
]

In [ ]:
all_calibration_ids = []

for prompt in calibration_prompts:
    token_ids = tokenizer.encode(
        prompt,
        add_special_tokens=False,
    )

    all_calibration_ids.extend(token_ids)
    all_calibration_ids.append(tokenizer.eos_token_id)


calibration_chunks = []

for start in range(
    0,
    len(all_calibration_ids) - 31,
    32,
):
    chunk = all_calibration_ids[
        start : start + 32
    ]

    calibration_chunks.append(chunk)


print("Number of calibration chunks:",
      len(calibration_chunks))

print("Tokens in each chunk:",
      len(calibration_chunks[0]))

In [ ]:
litert_exported_block0 = export_block(
    block0,
    block_input,
)

litert_quantizer = create_w8a8_quantizer()

litert_prepared_block0 = prepare_pt2e(
    litert_exported_block0.module(),
    litert_quantizer,
)

position_ids_32 = torch.arange(32).unsqueeze(0)

with torch.no_grad():
    for chunk in calibration_chunks:
        calibration_ids = torch.tensor(
            [chunk],
            dtype=torch.long,
        )

        calibration_block_input = (
            model.transformer.wte(calibration_ids)
            + model.transformer.wpe(position_ids_32)
        )

        litert_prepared_block0(
            calibration_block_input
        )

In [ ]:
litert_quantized_block0 = convert_pt2e(
    litert_prepared_block0,
    fold_quantize=False,
)

In [ ]:
import litert_torch

from litert_torch.quantize.quant_config import (
    QuantConfig,
)

sample_args = (
    block_input.detach(),
)

edge_block0 = litert_torch.convert(
    litert_quantized_block0,
    sample_args,
    quant_config=QuantConfig(
        pt2e_quantizer=litert_quantizer
    ),
)

## Compare FP32, PT2E, and LiteRT on a held-out prompt

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
test_prompt = 'The little dog walked into the forest'
test_encoded = tokenizer(test_prompt, return_tensors='pt', padding='max_length',
                         truncation=True, max_length=SEQUENCE_LENGTH)
new_input_ids = test_encoded['input_ids']
attention_mask = test_encoded['attention_mask']
new_block_input = create_block0_input(model, new_input_ids)

def as_numpy(output):
    if isinstance(output, (list, tuple)):
        output = output[0]
    if isinstance(output, torch.Tensor):
        output = output.detach().cpu().numpy()
    return np.asarray(output)

def error_metrics(reference, actual):
    assert reference.shape == actual.shape
    error = np.abs(reference - actual)
    return {'max_abs': float(error.max()), 'mean_abs': float(error.mean())}

with torch.no_grad():
    fp32_numpy = as_numpy(block0(new_block_input))
    pt2e_numpy = as_numpy(litert_quantized_block0(new_block_input))
litert_numpy = as_numpy(edge_block0(new_block_input.detach()))
mask = np.broadcast_to(attention_mask.numpy().astype(bool)[..., None], fp32_numpy.shape)
metrics = {
    'fp32_vs_pt2e': error_metrics(fp32_numpy, pt2e_numpy),
    'pt2e_vs_litert': error_metrics(pt2e_numpy, litert_numpy),
    'fp32_vs_litert': error_metrics(fp32_numpy, litert_numpy),
    'real_tokens_fp32_vs_litert': error_metrics(fp32_numpy[mask], litert_numpy[mask]),
}
print(json.dumps(metrics, indent=2))
# Quantization errors are reported, not automatically declared acceptable.

## Export, reload, and save provenance

In [ ]:
edge_block0.export(str(MODEL_PATH))
reloaded = litert_torch.load(str(MODEL_PATH))
reloaded_numpy = as_numpy(reloaded(new_block_input.detach()))
metrics['reload'] = error_metrics(litert_numpy, reloaded_numpy)
np.testing.assert_allclose(reloaded_numpy, litert_numpy, atol=1e-6, rtol=1e-6)

np.savez(EXPORT_DIR / 'block0_validation_inputs.npz',
         benchmark_input_ids=input_ids.numpy(),
         benchmark_block_input=block_input.numpy(),
         test_input_ids=new_input_ids.numpy(), attention_mask=attention_mask.numpy(),
         test_block_input=new_block_input.numpy(), fp32_output=fp32_numpy,
         pt2e_output=pt2e_numpy, litert_output=litert_numpy)
versions = {'python': platform.python_version()}
for package in ['torch', 'transformers', 'torchao', 'litert-torch', 'ai-edge-litert', 'numpy']:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = None
report = {
    'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'sequence_length': SEQUENCE_LENGTH,
    'model_sha256': hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest(),
    'versions': versions, 'calibration_prompts': calibration_prompts,
    'calibration_chunks': calibration_chunks, 'test_prompt': test_prompt, 'metrics': metrics,
    'quantizer': {'is_per_channel': False, 'is_dynamic': False, 'is_qat': False,
                  'fold_quantize': False},
}
(RESULT_DIR / 'block0_export.json').write_text(json.dumps(report, indent=2))
print('Saved:', MODEL_PATH)
print('Model bytes:', MODEL_PATH.stat().st_size)

In [ ]:
finish_run(ROOT, RUN_DIR, [MODEL_PATH, EXPORT_DIR / 'block0_validation_inputs.npz', RESULT_DIR / 'block0_export.json'])
print('Saved run record:', RUN_DIR / 'run.json')
